# 命令行版聊天应用

预计阅读时间：10-15 分钟。

本节逐步理解 [`06_CmdChat.py`](06_CmdChat.py)。命令行聊天应用的核心很简单：反复读取用户输入，调用模型，再打印回答。

## 从 while 循环开始

命令行程序需要一直等待用户输入，因此最外层通常是一个无限循环。在完整程序里，这个循环放在 `main()` 函数中：

In [2]:
def main() -> None:
    print("CmdChat 已启动。输入 exit 或 quit 退出。")
    while True:
        user_input = input("\n你：").strip()
        if user_input.lower() in {"exit", "quit"}:
            break
        if not user_input:
            continue
        print(f"\n助手：{ask(user_input)}")

这段循环完成四件事：

- `input()` 读取用户输入；
- 输入 `exit` 或 `quit` 时退出；
- 空输入直接跳过；
- 有效输入交给 `ask()`，再打印助手回答。

这里先定义 `main()`，不会立刻执行。后面定义好 `ask()` 后，再调用 `main()`。

## 导入依赖

接着导入本程序需要的库：

In [3]:
import os

from dotenv import load_dotenv
from openai import OpenAI

## 准备客户端

调用模型前，需要读取 API Key，并创建 OpenAI 兼容客户端：

In [4]:
load_dotenv()

api_key = os.getenv("ZAI_API_KEY")
if not api_key:
    raise RuntimeError("请先设置环境变量 ZAI_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://open.bigmodel.cn/api/paas/v4/",
)
MODEL_NAME = "glm-5.2"


这里沿用前面章节的约定：API Key 放在环境变量或 `.env` 文件中，不写进代码。

## system prompt

`SYSTEM_PROMPT` 用来设定助手的整体行为：

In [5]:
SYSTEM_PROMPT = (
    "你是一个通用学习助手。回答要简洁，注重原理阐释和示例引导；"
    "优先通过苏格拉底式提问启发用户思考。"
)

它不是用户的具体问题，而是对模型回答风格的长期约束。

## 保存历史

多轮聊天需要保存历史。最开始只有一条 `system` 消息：

In [6]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
]

每一轮对话都会继续往 `messages` 中追加内容。

## ask 函数

`ask()` 是命令行聊天应用的核心函数：

In [7]:
def ask(user_input: str) -> str:
    messages.append({"role": "user", "content": user_input})
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=0.6,
    )
    answer = response.choices[0].message.content
    messages.append({"role": "assistant", "content": answer})
    return answer

它的流程如下：

```mermaid
flowchart TD
    A[用户输入 user_input] --> B[追加到 messages 历史]
    B --> C[把完整 messages 发给模型]
    C --> D[取出模型回答 answer]
    D --> E[把 answer 追加到 messages 历史]
    E --> F[返回 answer 文本]
```



## 为什么要保存历史

保存历史是多轮对话的基础。否则模型每次只能看到当前输入，无法知道前面聊过什么，也看不到自己之前的回答。

例如：

```text
你：请解释递归。
助手：递归是函数调用自身……
你：用刚才的概念解释斐波那契。
```

第二个问题里的“刚才”依赖历史消息。如果没有历史，模型就无法准确理解这个指代。

In [8]:
example_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "请解释递归。"},
    {"role": "assistant", "content": "递归是函数调用自身……"},
    {"role": "user", "content": "用刚才的概念解释斐波那契。"},
]

服务端会把这些消息组织成模型可以理解的上下文。粗略理解，可以想象成下面这种拼接形式：

```text
<system> 你是一个通用学习助手……
<user> 请解释递归。
<assistant> 递归是函数调用自身……
<user> 用刚才的概念解释斐波那契。
<assistant>
```

实际 API 可能使用内部格式，但核心不变：`system`、`user`、`assistant` 消息共同构成当前上下文。大模型会基于这个上下文继续预测下一个 token。

## 入口代码

最后调用 `main()` 启动程序：

In [10]:
if __name__ == "__main__":
    main()

CmdChat 已启动。输入 exit 或 quit 退出。

助手：这是一个最基础的 Python "Hello World" 程序：

```python
print("Hello, World!")
```

**原理阐释：**
* `print()` 是 Python 的内置函数，它的作用是将括号里的内容输出到屏幕（控制台）上。
* `"Hello, World!"` 是一个字符串（由双引号括起来的文本），代表你想要输出的具体信息。

**启发思考：**
如果你想让程序向特定的名字问好，比如输出 `"Hello, Alice!"`，你觉得代码应该怎么修改？如果要实现“让用户自己输入名字，程序再向其问好”，你猜需要用到什么新的概念呢？


这表示：直接运行 `python3 06_CmdChat.py` 时启动聊天；如果这个文件被其他程序导入，则不会自动运行。

在终端中，也可以直接运行完整脚本：

```bash
python3 06_CmdChat.py
```

输入问题后按回车。输入 `exit` 或 `quit` 退出。

## 小结

命令行版只做最少的事情：

- 用 `while True` 持续读取输入；
- 读取 API Key 并创建客户端；
- 用 `messages` 保存多轮历史；
- 用 `ask()` 封装一次模型调用；
- 用 `main()` 组织程序入口。